In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
#GWAS_SNPs_df = pd.read_csv("../hg38_combined_unique_DCM_SNPs.txt", sep="\t")
GWAS_SNPs_df = pd.read_csv("../03_combined_SNP_list_with_effector_genes.csv", index_col = 0)
#GWAS_SNPs_df.columns = ["SNP"]

In [3]:
GWAS_SNPs_df

,chr,start,end,rsID,candidate_gene,total_score,study,locus,plink_name
0,chr1,2212668,2212668,rs2503715,SKI_or_C1orf86,2,both,SKI_or_C1orf86_locus0,1:2212668
1,chr1,3324690,3324690,rs79548216,PRDM16,3,Zheng,PRDM16_locus0,1:3324690
2,chr1,6203732,6203732,rs11121483,RNF207,3,Jurgens,RNF207_locus0,1:6203732
3,chr1,6218354,6218354,rs709209,RNF207,3,Zheng,RNF207_locus0,1:6218354
4,chr1,16012430,16012430,rs1763605,HSPB7,2,Jurgens,HSPB7_locus0,1:16012430
...,...,...,...,...,...,...,...,...,...
152,chr21,29199348,29199348,rs8134232,BACH1,2,Zheng,BACH1_locus0,21:29199348
153,chr21,38657121,38657121,rs796217035,ERG,3,Zheng,ERG_locus0,21:38657121
154,chr21,39272244,39272244,rs8134638,BRWD1,2,Jurgens,BRWD1_locus0,21:39272244
155,chr22,23819530,23819530,rs5760054,DERL3,3,Jurgens,DERL3_locus0,22:23819530


In [4]:
num_SNPs = GWAS_SNPs_df.shape[0]
print(num_SNPs)

157


In [5]:
num_loci = len(GWAS_SNPs_df['locus'].unique())
print(num_loci)

117


In [6]:
all_res_list = list()

for SNP in GWAS_SNPs_df['plink_name']:

    SNP_res_path = "LD_results/" + SNP + ".ld"

    if os.path.exists(SNP_res_path):
        SNP_results_df = pd.read_csv("LD_results/" + SNP + ".ld", delim_whitespace=True)
        all_res_list.append(SNP_results_df)

all_res_df = pd.concat(all_res_list).reset_index(drop=True)

### These are all SNPs within 1 Mb, we will filter this to those with R^2 > 0.2

In [7]:
all_res_df

,CHR_A,BP_A,SNP_A,CHR_B,BP_B,SNP_B,R2
0,1,2212668,1:2212668:A:G,1,2212668,1:2212668:A:G,1.000000
1,1,2212668,1:2212668:A:G,1,2213543,1:2213543:G:T,0.336155
2,1,2212668,1:2212668:A:G,1,2215707,1:2215707:T:C,0.244966
3,1,2212668,1:2212668:A:G,1,2225467,1:2225468:A:AT,0.244404
4,1,2212668,1:2212668:A:G,1,2234971,1:2234971:T:C,0.245585
...,...,...,...,...,...,...,...
30531,22,23824601,22:23824601:G:A,22,23839465,22:23839465:A:G,0.395402
30532,22,23824601,22:23824601:G:A,22,23839987,22:23839987:C:CTGGGAAGAT,0.468384
30533,22,23824601,22:23824601:G:A,22,23840727,22:23840728:CA:C,0.302268
30534,22,23824601,22:23824601:G:A,22,23841011,22:23841011:C:T,0.345468


In [8]:
all_res_df['distance'] = np.abs(all_res_df['BP_B'] - all_res_df['BP_A'])

In [10]:
LD_SNP_df = all_res_df[all_res_df['R2'] > 0.8].reset_index(drop=True)
print(LD_SNP_df.shape)

(3046, 8)


In [11]:
LD_SNP_df['lead_SNP'] = ( LD_SNP_df['CHR_A'].astype(str) + ":" +
                         LD_SNP_df['BP_A'].astype(str))

In [12]:
merged_df = LD_SNP_df.merge(GWAS_SNPs_df, left_on = "lead_SNP", right_on = "plink_name")
merged_df.head()

,CHR_A,BP_A,SNP_A,CHR_B,BP_B,SNP_B,R2,distance,lead_SNP,chr,start,end,rsID,candidate_gene,total_score,study,locus,plink_name
0,1,2212668,1:2212668:A:G,1,2212668,1:2212668:A:G,1.000000,0,1:2212668,chr1,2212668,2212668,rs2503715,SKI_or_C1orf86,2,both,SKI_or_C1orf86_locus0,1:2212668
1,1,3324690,1:3324690:A:C,1,3324690,1:3324690:A:C,1.000000,0,1:3324690,chr1,3324690,3324690,rs79548216,PRDM16,3,Zheng,PRDM16_locus0,1:3324690
2,1,3324690,1:3324690:A:C,1,3324727,1:3324727:T:C,0.828899,37,1:3324690,chr1,3324690,3324690,rs79548216,PRDM16,3,Zheng,PRDM16_locus0,1:3324690
3,1,3324690,1:3324690:A:C,1,3328231,1:3328231:G:A,0.853674,3541,1:3324690,chr1,3324690,3324690,rs79548216,PRDM16,3,Zheng,PRDM16_locus0,1:3324690
4,1,3324690,1:3324690:A:C,1,3329549,1:3329549:G:C,0.853674,4859,1:3324690,chr1,3324690,3324690,rs79548216,PRDM16,3,Zheng,PRDM16_locus0,1:3324690


In [20]:
len(merged_df['locus'].unique())

116

In [12]:
merged_df.to_csv("04_lead_and_LD_SNPs.csv", index=False)

### Save in a bed format

In [13]:
LD_SNP_bed_df = LD_SNP_df[["CHR_A", "BP_B", "BP_B", "SNP_A"]].copy()
LD_SNP_bed_df['CHR_A'] = "chr" + LD_SNP_bed_df['CHR_A'].astype(str)

In [14]:
LD_SNP_bed_df.to_csv("04_lead_and_LD_SNPs.bed", sep = "\t", header=False, index=False)

### Calculate the max distance between a lead SNP and another SNP with R^2>0.8

In [15]:
distances = np.abs(LD_SNP_df['BP_B'] - LD_SNP_df['BP_A'])
np.max(distances)

564943